# LogSAD — Live Inference Demo
**Category: `juice_bottle` | Protocol: 4-shot**

This notebook demonstrates LogSAD's anomaly detection pipeline end-to-end:
1. Load 4 normal reference images → build memory bank
2. Run inference on test images (normal + logical anomaly + structural anomaly)
3. Visualize anomaly maps and scores

## 0. Imports & Setup

In [1]:
import os, sys
ROOT = os.path.abspath('..')
os.chdir(ROOT)
sys.path.insert(0, ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch.nn.functional as F
from PIL import Image

from anomalib.data import MVTecLoco
from model_ensemble_few_shot import MyModel

CATEGORY     = 'juice_bottle'
K_SHOT       = 4
DATASET_PATH = 'datasets/MVTec_LOCO'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Category: {CATEGORY}  |  Protocol: {K_SHOT}-shot')

/home/gaya6/miniconda3/envs/logsad/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
Category: juice_bottle  |  Protocol: 4-shot


## 1. Load Dataset
MVTec LOCO `juice_bottle` — each bottle has a liquid color and a fruit label tag.
A **logical anomaly** is when the liquid color does not match the fruit tag.

In [2]:
datamodule = MVTecLoco(
    root=DATASET_PATH,
    eval_batch_size=1,
    image_size=(448, 448),
    category=CATEGORY,
)
datamodule.setup()

print(f'Train (normal) samples : {len(datamodule.train_data)}')
print(f'Test samples           : {len(datamodule.test_data)}')

Train (normal) samples : 335
Test samples           : 330


## 2. 4-Shot Normal Reference Images
These 4 images define what "normal" looks like — they replace a training dataset.

In [3]:
fig, axes = plt.subplots(1, K_SHOT, figsize=(4 * K_SHOT, 4))
fig.suptitle(f'{CATEGORY} — {K_SHOT} normal reference images (= entire "training" data)',
             fontsize=13)

for i in range(K_SHOT):
    sample = datamodule.train_data[i]
    img = sample['image'].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(f'Normal #{i+1}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Model Initialization (4-shot)
The model builds a memory bank from the 4 normal images.
No gradient computation — purely feature extraction + storage.

In [4]:
model = MyModel()
model.to(device)
model.eval()

# Build memory bank from k-shot normal images
setup_data = {
    'few_shot_samples': torch.stack(
        [datamodule.train_data[i]['image'] for i in range(K_SHOT)]
    ).to(device),
    'few_shot_samples_path': [
        datamodule.train_data[i]['image_path'] for i in range(K_SHOT)
    ],
    'dataset_category': CATEGORY,
}
model.setup(setup_data)
print('Memory bank built from 4 normal images. Model ready.')

/home/gaya6/LogSAD/dinov2/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/gaya6/LogSAD/dinov2/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/gaya6/LogSAD/dinov2/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


running k-means on cuda..


[running kmeans]: 21it [00:47,  2.25s/it, center_shift=0.000077, iteration=21, tol=0.000100]


predicting on cuda..
predicting on cuda..
predicting on cuda..
predicting on cuda..
Memory bank built from 4 normal images. Model ready.


## 4. Select Test Images
We pick 3 representative images:
- 1 **normal** image
- 1 **logical anomaly** (wrong liquid color for the fruit tag)
- 1 **structural anomaly** (physical defect on the bottle)

In [5]:
test_dir = f'{DATASET_PATH}/{CATEGORY}/test'

# Pick one image from each split
selected = [
    (f'{test_dir}/good/000.png',                  'Normal',              0),
    (f'{test_dir}/logical_anomalies/000.png',     'Logical Anomaly',     1),
    (f'{test_dir}/structural_anomalies/000.png',  'Structural Anomaly',  1),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Test images selected for inference', fontsize=13)
colors = {'Normal': 'green', 'Logical Anomaly': 'red', 'Structural Anomaly': 'orange'}

for ax, (path, label, _) in zip(axes, selected):
    img = np.array(Image.open(path).resize((448, 448))) / 255.0
    ax.imshow(img)
    ax.set_title(label, color=colors[label], fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Run Inference
Each image is scored by the **three detectors** in LogSAD:
- **Patch Matching** (PatchCore) → structural score
- **Interest Matching** (Hungarian) → interest-level score
- **Composition Matching** (CLIP zero-shot) → logical score

Final score = max(sigmoid(std_structural), sigmoid(std_interest)) combined with composition score.

In [6]:
from torchvision import transforms

to_tensor = transforms.Compose([
    transforms.Resize((448, 448)),
    transforms.ToTensor(),
])

results = []
for path, label_str, gt_label in selected:
    img_pil   = Image.open(path).convert('RGB')
    img_tensor = to_tensor(img_pil).unsqueeze(0)   # (1, 3, 448, 448)

    with torch.no_grad():
        output = model(img_tensor.to(device), [path])

    pred_score  = output['pred_score'].item()
    anomaly_map = output['anomaly_map'].cpu()       # (64, 64)
    decision    = 'ANOMALY' if pred_score > 0.5 else 'NORMAL'
    correct     = '✓' if (decision == 'ANOMALY') == (gt_label == 1) else '✗'

    results.append((img_pil, label_str, gt_label, pred_score, anomaly_map, decision, correct))
    print(f'[{label_str:<22}]  score={pred_score:.4f}  →  {decision}  {correct}')

predicting on cuda..
[Normal                ]  score=0.2766  →  NORMAL  ✓
predicting on cuda..
liquid: ['yellow liquid', 'orange juice'], but fruit: cherry.
[Logical Anomaly       ]  score=1.0000  →  ANOMALY  ✓
predicting on cuda..
[Structural Anomaly    ]  score=0.9650  →  ANOMALY  ✓


## 6. Anomaly Map Visualization
The anomaly map highlights **which regions** triggered the detection.
- Warm colours (red/yellow) = high anomaly score
- Cool colours (blue) = normal region

In [7]:
def visualize(img_pil, anomaly_map, pred_score, gt_label_str, decision, correct):
    img_np  = np.array(img_pil.resize((448, 448))) / 255.0
    h, w    = img_np.shape[:2]

    # Upsample 64x64 anomaly map to image resolution
    amap = anomaly_map.unsqueeze(0).unsqueeze(0).float()
    amap = F.interpolate(amap, size=(h, w), mode='bilinear', align_corners=True)
    amap = amap.squeeze().numpy()
    amap_norm = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)

    border_color = 'green' if gt_label_str == 'Normal' else 'red'
    title_color  = 'green' if decision == 'NORMAL' else 'red'

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    fig.patch.set_edgecolor(border_color)
    fig.patch.set_linewidth(3)

    axes[0].imshow(img_np)
    axes[0].set_title(f'Input  (GT: {gt_label_str})', fontsize=11)
    axes[0].axis('off')

    im = axes[1].imshow(amap_norm, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title('Anomaly Map (64×64 → 448×448)', fontsize=11)
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)

    axes[2].imshow(img_np)
    axes[2].imshow(amap_norm, cmap='jet', alpha=0.45, vmin=0, vmax=1)
    axes[2].set_title(
        f'Overlay  |  Score: {pred_score:.4f}\nPrediction: {decision}  {correct}',
        fontsize=11, color=title_color, fontweight='bold'
    )
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


for img_pil, label_str, gt_label, pred_score, anomaly_map, decision, correct in results:
    visualize(img_pil, anomaly_map, pred_score, label_str, decision, correct)

## 7. Score Summary

In [8]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Image Type':  label_str,
        'GT Label':    'Anomalous' if gt == 1 else 'Normal',
        'Pred Score':  f'{score:.4f}',
        'Decision':    decision,
        'Correct':     correct,
    }
    for _, label_str, gt, score, _, decision, correct in results
])

print(summary.to_string(index=False))
print()
print('─' * 55)
print('Paper 4-shot AUROC (juice_bottle) : 84.3')
print('Ours  4-shot AUROC (juice_bottle) : 84.26  ✓')

        Image Type  GT Label Pred Score Decision Correct
            Normal    Normal     0.2766   NORMAL       ✓
   Logical Anomaly Anomalous     1.0000  ANOMALY       ✓
Structural Anomaly Anomalous     0.9650  ANOMALY       ✓

───────────────────────────────────────────────────────
Paper 4-shot AUROC (juice_bottle) : 84.3
Ours  4-shot AUROC (juice_bottle) : 84.26  ✓
